# Jour 3 — Construction et entraînement du CNN 

**Objectif du jour :** construire un réseau de neurones convolutif (CNN), l'entraîner et lire ses courbes.


## Étape 1 — Importer et configurer

**Problème :** préparer les outils et les paramètres .

**Explication du code :**
- `MODELS_DIR` : dossier où l'on sauvegardera le modèle (sur Kaggle : `/kaggle/working/models`).
- Si le dossier `split` du Jour 1 n'existe pas (nouvelle session), le code le cherche dans les Inputs.
- `tf.config.list_physical_devices("GPU")` vérifie que le GPU est bien actif.

In [9]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

CLASSES = ["buildings", "forest", "glacier", "mountain", "sea", "street"]
IMG_SIZE = (150, 150)
BATCH_SIZE = 32
SEED = 42

if os.path.exists("/kaggle/working"):
    SPLIT_DIR = Path("/kaggle/working/data/split")
    MODELS_DIR = Path("/kaggle/working/models")
else:
    SPLIT_DIR = Path("data/split")
    MODELS_DIR = Path("models")

# Nouvelle session Kaggle : chercher le split dans les Inputs
if not SPLIT_DIR.exists() and os.path.exists("/kaggle/input"):
    for dossier, sous_dossiers, fichiers in os.walk("/kaggle/input"):
        if dossier.endswith("split") and "train" in sous_dossiers:
            SPLIT_DIR = Path(dossier)
            break

assert SPLIT_DIR.exists(), "Dossier split introuvable : exécutez d'abord le Jour 1."
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Dossier split :", SPLIT_DIR)
print("GPU disponible :", tf.config.list_physical_devices("GPU"))

Dossier split : data/split
GPU disponible : []


## Étape 2 — Charger les données


In [10]:
train_ds = keras.utils.image_dataset_from_directory(
    str(SPLIT_DIR / "train"), label_mode="int", class_names=CLASSES,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=True, seed=SEED)

val_ds = keras.utils.image_dataset_from_directory(
    str(SPLIT_DIR / "val"), label_mode="int", class_names=CLASSES,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False)

test_ds = keras.utils.image_dataset_from_directory(
    str(SPLIT_DIR / "test"), label_mode="int", class_names=CLASSES,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False)

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

Found 11930 files belonging to 6 classes.
Found 2552 files belonging to 6 classes.
Found 2552 files belonging to 6 classes.


## Étape 3 — Construire le CNN

**Problème :** une image 150×150×3 contient 67 500 nombres. Un réseau classique ne tient pas compte du fait que **les pixels voisins sont liés**.
Le CNN, lui, cherche des **motifs locaux** (bords, textures, formes).


In [11]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1,0.1),
    layers.RandomContrast(0.1),
])

model = keras.Sequential([
    keras.Input(shape=(150,150,3)),
    data_augmentation,
    layers.Rescaling(1.0 / 255),
    layers.Conv2D(32,(3,3),activation="relu",padding="same"),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64,(3,3),activation="relu",padding="same"),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(128,(3,3),activation="relu",padding="same"),
    layers.MaxPooling2D((2,2)),
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(6,activation="softmax")
])
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential_2 (Sequential)       │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_1 (Rescaling)         │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 150, 150, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 75, 75, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 75, 75, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 37, 37, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 37, 37, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 18, 18, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 110,534 (431.77 KB)

 Trainable params: 110,534 (431.77 KB)

 Non-trainable params: 0 (0.00 B)

## Étape 4 — Compiler le modèle
**Problème :** dire au modèle **comment mesurer ses erreurs** et **comment s'améliorer**.


In [12]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

## Étape 5 — Entraîner le modèle

**Problème :** combien d'époques ? Trop peu : le modèle n'a pas assez appris. Trop : il mémorise (overfitting).

**Explication du code :**
- Une **époque** = un passage complet sur toutes les images d'entraînement. On autorise 30 époques maximum.
- **EarlyStopping** : arrête l'entraînement si la `val_loss` ne s'améliore plus pendant 5 époques, et **remet les meilleurs poids** (`restore_best_weights=True`).
- **ReduceLROnPlateau** : si la `val_loss` stagne pendant 2 époques, le learning rate est divisé par 2 (pas plus petits = réglage plus fin).
- `validation_data=val_ds` : à la fin de chaque époque, le modèle est testé sur la validation (le modèle n'apprend **pas** dessus).
- `history` garde les valeurs de chaque époque pour tracer les courbes.

In [13]:
arret_precoce = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True)

reduire_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=2)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=[arret_precoce, reduire_lr],
)

Epoch 1/30
373/373 ━━━━━━━━━━━━━━━━━━━━ 107s 285ms/step - accuracy: 0.4176 - loss: 1.3575 - val_accuracy: 0.3907 - val_loss: 1.6759 - learning_rate: 0.0010
Epoch 2/30
373/373 ━━━━━━━━━━━━━━━━━━━━ 112s 300ms/step - accuracy: 0.5422 - loss: 1.1267 - val_accuracy: 0.5263 - val_loss: 1.1756 - learning_rate: 0.0010
Epoch 3/30
373/373 ━━━━━━━━━━━━━━━━━━━━ 114s 306ms/step - accuracy: 0.5790 - loss: 1.0479 - val_accuracy: 0.4992 - val_loss: 1.3253 - learning_rate: 0.0010
Epoch 4/30
373/373 ━━━━━━━━━━━━━━━━━━━━ 119s 319ms/step - accuracy: 0.6065 - loss: 0.9911 - val_accuracy: 0.6328 - val_loss: 0.9174 - learning_rate: 0.0010
Epoch 5/30
373/373 ━━━━━━━━━━━━━━━━━━━━ 111s 296ms/step - accuracy: 0.6431 - loss: 0.9295 - val_accuracy: 0.5925 - val_loss: 1.0827 - learning_rate: 0.0010
Epoch 6/30
165/373 ━━━━━━━━━━━━━━━━━━━━ 1:00 292ms/step - accuracy: 0.6479 - loss: 0.9081

KeyboardInterrupt: 

## Étape 6 — Courbes Loss et Accuracy

**Explication du code :** `history.history` est un dictionnaire avec 4 listes :
`loss`, `accuracy` (train) et `val_loss`, `val_accuracy` (validation). On trace les deux courbes de chaque mesure sur le même graphique.


In [ ]:
plt.figure(figsize=(12, 4))

# Loss
plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="validation")
plt.title("Loss")
plt.xlabel("époque")
plt.legend()

# Accuracy
plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="validation")
plt.title("Accuracy")
plt.xlabel("époque")
plt.legend()

plt.show()

## Étape 7 — Analyser les résultats


In [ ]:
hist = pd.DataFrame(history.history)
meilleure = hist["val_loss"].idxmin()

acc_train = hist.loc[meilleure, "accuracy"]
acc_val = hist.loc[meilleure, "val_accuracy"]

print("Époques effectuées      :", len(hist))
print("Meilleure époque        :", meilleure + 1)
print("Accuracy train          :", round(acc_train, 3))
print("Accuracy validation     :", round(acc_val, 3))
print("Écart train - validation:", round(acc_train - acc_val, 3))

## Étape 8 — Sauvegarder le modèle

**Problème :** garder le modèle entraîné pour le Jour 4 (évaluation).

**Explication du code :**
- `model.save` enregistre l'architecture **et** les poids dans un fichier `.keras`.
- On sauvegarde aussi l'historique (`history_cnn_v1.csv`) pour retracer les courbes plus tard.

In [ ]:
model.save(str(MODELS_DIR / "cnn_v1.keras"))
hist.to_csv(MODELS_DIR / "history_cnn_v1.csv", index=False)

print("Fichiers sauvegardés :", os.listdir(MODELS_DIR))